# 18yB — Dissertation output freeze

This stage converts the verified 18yA evidence registry into final
dissertation-facing artefacts:

- five `booktabs` tables;
- six synthesis figures;
- reusable LaTeX macros;
- concise Results and Discussion statements;
- a figure and table placement inventory.

It depends only on the frozen 18yA synthesis release and performs no model,
calibration or trading selection.

**Revision v2.** This presentation-only patch bolds the lowest probability scores within each evaluation block, discloses the omitted uncalibrated CatBoost rows in the compact table and figures, corrects the sample-flow caption, and states the contract-book structure precisely. No empirical selection or numerical result is recomputed.

In [1]:
from __future__ import annotations

import hashlib
import json
import platform
import re
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / '.git').exists():
    raise RuntimeError(f'Run this notebook from the repository root, not {ROOT}')

UTC = timezone.utc
STEP = '18yB'

A_DIR = ROOT / 'data/processed/18yA_final_empirical_synthesis'
A_REPORT_DIR = ROOT / 'reports/18yA_final_empirical_synthesis'
OUT_DIR = ROOT / 'data/processed/18yB_thesis_output_freeze'
REPORT_DIR = ROOT / 'reports/18yB_thesis_output_freeze'
FIGURE_DIR = REPORT_DIR / 'figures'
OUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

A_SUMMARY = A_DIR / '18yA_summary.json'
A_MANIFEST = A_DIR / '18yA_sha256_manifest.csv'
A_SAMPLE = A_DIR / '18yA_final_sample_flow.csv'
A_WEATHER = A_DIR / '18yA_deterministic_forecast_diagnostics.csv'
A_MODELS = A_DIR / '18yA_development_model_selection.csv'
A_CALIBRATION = A_DIR / '18yA_calibration_selection.csv'
A_PROBABILITY = A_DIR / '18yA_probability_evaluation_compact.csv'
A_PROBABILITY_FULL = A_DIR / '18yA_probability_evaluation_full.csv'
A_TRADING_SELECTION = A_DIR / '18yA_trading_strategy_selection.csv'
A_TRADING = A_DIR / '18yA_trading_evaluation.csv'
A_CLAIMS = A_DIR / '18yA_verified_claims.csv'
A_BOUNDARIES = A_DIR / '18yA_evidential_boundaries.csv'

REQUIRED = [
    A_SUMMARY, A_MANIFEST, A_SAMPLE, A_WEATHER, A_MODELS,
    A_CALIBRATION, A_PROBABILITY, A_PROBABILITY_FULL, A_TRADING_SELECTION,
    A_TRADING, A_CLAIMS, A_BOUNDARIES,
]
for path in REQUIRED:
    if not path.is_file():
        raise FileNotFoundError(f'Required 18yA input is missing: {path}')

In [2]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def verify_manifest(path: Path) -> int:
    manifest = pd.read_csv(path)
    failures = []
    for row in manifest.itertuples(index=False):
        candidate = ROOT / row.path
        if not candidate.is_file():
            failures.append(f'MISSING: {row.path}')
            continue
        if sha256_file(candidate) != row.sha256:
            failures.append(f'HASH: {row.path}')
        if candidate.stat().st_size != int(row.size_bytes):
            failures.append(f'SIZE: {row.path}')
    if failures:
        raise AssertionError('18yA manifest verification failed:\n' + '\n'.join(failures))
    return len(manifest)


def tex_escape(value: Any) -> str:
    text = str(value)
    replacements = {
        '\\': r'\textbackslash{}',
        '&': r'\&',
        '%': r'\%',
        '$': r'\$',
        '#': r'\#',
        '_': r'\_',
        '{': r'\{',
        '}': r'\}',
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    return text


def fmt(value: float, digits: int = 3) -> str:
    return f'{float(value):.{digits}f}'


manifest_entries = verify_manifest(A_MANIFEST)
summary = json.loads(A_SUMMARY.read_text(encoding='utf-8'))
if summary.get('verdict') != 'PASS':
    raise AssertionError('18yA is not a PASS release.')

sample = pd.read_csv(A_SAMPLE)
weather = pd.read_csv(A_WEATHER)
models = pd.read_csv(A_MODELS)
calibration = pd.read_csv(A_CALIBRATION)
probability = pd.read_csv(A_PROBABILITY)
probability_full = pd.read_csv(A_PROBABILITY_FULL)
trading_selection = pd.read_csv(A_TRADING_SELECTION)
trading = pd.read_csv(A_TRADING)
claims = pd.read_csv(A_CLAIMS)
boundaries = pd.read_csv(A_BOUNDARIES)

key = summary['key_metrics']

if (len(models) != 9 or len(calibration) != 3 or len(probability) != 12 or len(probability_full) != 14 or len(trading) != 6):
    raise AssertionError('Unexpected 18yA synthesis table dimensions.')

print('Verified 18yA synthesis release: PASS')

Verified 18yA synthesis release: PASS


In [3]:
# ------------------------------------------------------------------
# LaTeX macros
# ------------------------------------------------------------------
macro_values = {
    'NSettlementDates': str(int(key['certified_dates'])),
    'NCertifiedContracts': str(int(key['certified_contracts'])),
    'NDevelopmentDates': str(int(key['development_dates'])),
    'NCommonSelectionRows': str(int(key['common_selection_rows'])),
    'NHoldoutDates': str(int(key['holdout_dates'])),
    'NExternalDates': str(int(key['external_dates'])),
    'DevelopmentEmpiricalCRPS': fmt(key['development_empirical_crps']),
    'DevelopmentRawCRPS': fmt(key['development_raw_crps']),
    'DevelopmentEmpiricalCRPSReduction': fmt(key['development_empirical_crps_reduction_pct'], 1),
    'HoldoutEmpiricalLogScore': fmt(key['holdout_empirical_uncalibrated_categorical_log']),
    'HoldoutMarketLogScore': fmt(key['holdout_market_categorical_log']),
    'ExternalMarketLogScore': fmt(key['external_market_categorical_log']),
    'ExternalBestModelLogScore': fmt(key['external_best_model_categorical_log']),
    'PrimaryHoldoutPnL': fmt(key['primary_holdout_net_pnl'], 4),
    'PrimaryExternalPnL': fmt(key['primary_external_net_pnl'], 4),
    'PrimaryHoldoutDrawdown': fmt(key['primary_holdout_max_drawdown'], 4),
    'PrimaryExternalDrawdown': fmt(key['primary_external_max_drawdown'], 4),
}

macros_lines = [
    '% Auto-generated by 18yB. Values are frozen to the verified 18yA release.',
]
for name, value in macro_values.items():
    macros_lines.append(rf'\newcommand{{\{name}}}{{{value}}}')
macros_lines.append('')
macros_text = '\n'.join(macros_lines)
macros_path = REPORT_DIR / '18yB_results_macros.tex'
macros_path.write_text(macros_text, encoding='utf-8')

# ------------------------------------------------------------------
# LaTeX tables
# ------------------------------------------------------------------
sample_rows = sample.loc[
    sample['object'].isin(
        [
            'Certified settlement dates',
            'Certified contracts',
            'Contract-decision candidates',
            'Market-price-ready rows',
            'Deterministic-weather-ready rows',
            'Exact market-weather common support',
            'Complete common-support books',
            'Development model-ready support',
            'Common OOF selection support',
            'Holdout model-ready support',
            'June model-ready support',
            'Exact market-common holdout books',
            'Exact market-common June books',
        ]
    )
].copy()

table_1 = [
    r'\begin{table}[htbp]',
    r'\centering',
    r'\caption{Final empirical sample flow and evaluation support.}',
    r'\label{tab:final_sample_flow}',
    r'\begin{tabular}{lrl}',
    r'\toprule',
    r'Object & Count & Unit \\',
    r'\midrule',
]
for row in sample_rows.itertuples(index=False):
    table_1.append(f'{tex_escape(row.object)} & {int(row.count):,} & {tex_escape(row.unit)} \\\\')
table_1.extend([r'\bottomrule', r'\end{tabular}', r'\end{table}', ''])

table_2 = [
    r'\begin{table}[htbp]',
    r'\centering',
    r'\caption{Development-only comparison of residual post-processing candidates. Lower CRPS and MAE are preferred.}',
    r'\label{tab:development_model_selection}',
    r'\begin{tabular}{rlrrr}',
    r'\toprule',
    r'Rank & Candidate & CRPS & MAE & 80\% coverage \\',
    r'\midrule',
]
for row in models.sort_values('overall_rank').itertuples(index=False):
    label = tex_escape(row.candidate_label)
    if bool(row.selected_family_representative):
        label = r'\textbf{' + label + '}'
    table_2.append(
        f'{int(row.overall_rank)} & {label} & {row.date_balanced_mean_crps_c:.3f} & '
        f'{row.date_balanced_mean_absolute_error_c:.3f} & {100.0 * row.date_balanced_interval_80_coverage:.1f}\% \\\\'
    )
table_2.extend([r'\bottomrule', r'\end{tabular}', r'\end{table}', ''])

table_3 = [
    r'\begin{table}[htbp]',
    r'\centering',
    r'\caption{Development-selected calibration parameters.}',
    r'\label{tab:calibration_selection}',
    r'\begin{tabular}{lrrrr}',
    r'\toprule',
    r'Candidate & Scale & Uniform mix & Log score & Multiclass Brier \\',
    r'\midrule',
]
for row in calibration.sort_values('candidate_label').itertuples(index=False):
    table_3.append(
        f'{tex_escape(row.candidate_label)} & {row.scale_factor:.2f} & {row.uniform_smoothing:.3f} & '
        f'{row.date_balanced_mean_categorical_log_score:.3f} & {row.date_balanced_mean_multiclass_brier:.3f} \\\\'
    )
table_3.extend([r'\bottomrule', r'\end{tabular}', r'\end{table}', ''])

probability = probability.copy()
probability['best_log_in_block'] = probability.groupby('evaluation_block')['categorical_log_score'].transform(
    lambda values: np.isclose(values, values.min(), rtol=0.0, atol=1e-12)
)
probability['best_brier_in_block'] = probability.groupby('evaluation_block')['multiclass_brier'].transform(
    lambda values: np.isclose(values, values.min(), rtol=0.0, atol=1e-12)
)

uncalibrated_catboost = probability_full.loc[
    probability_full['candidate_id'].eq('catboost_quantile_pooled')
    & probability_full['probability_variant'].eq('UNCALIBRATED')
].set_index('evaluation_block')
catboost_holdout_log = float(uncalibrated_catboost.loc['INTERNAL_HOLDOUT', 'categorical_log_score'])
catboost_external_log = float(uncalibrated_catboost.loc['EXTERNAL_TEST', 'categorical_log_score'])

table_4 = [
    r'\begin{table}[htbp]',
    r'\centering',
    r'\caption{Exact-common probability evaluation against the normalised market. Lower scores are preferred.}',
    r'\label{tab:probability_evaluation}',
    r'\begin{tabular}{llrr}',
    r'\toprule',
    r'Block & Method & Categorical log & Multiclass Brier \\',
    r'\midrule',
]
for row in probability.itertuples(index=False):
    method = tex_escape(row.method_label)
    log_value = f'{row.categorical_log_score:.3f}'
    brier_value = f'{row.multiclass_brier:.3f}'
    if bool(row.best_log_in_block) and bool(row.best_brier_in_block):
        method = r'\textbf{' + method + '}'
    if bool(row.best_log_in_block):
        log_value = r'\textbf{' + log_value + '}'
    if bool(row.best_brier_in_block):
        brier_value = r'\textbf{' + brier_value + '}'
    table_4.append(
        f'{tex_escape(row.evaluation_block_label)} & {method} & '
        f'{log_value} & {brier_value} \\\\'
    )
table_4.extend(
    [
        r'\bottomrule',
        r'\end{tabular}',
        r'\begin{minipage}{0.94\textwidth}',
        r'\footnotesize Bold denotes the lowest score within each evaluation block. The internal holdout contains ten settlement dates and is interpreted descriptively; the June block contains thirty dates and is the principal external evaluation. '
        + rf'The uncalibrated CatBoost rows are omitted from this compact display because zero winning probabilities produce categorical log scores {catboost_holdout_log:.3f} on the holdout and {catboost_external_log:.3f} in June; they remain in the full audit table.',
        r'\end{minipage}',
        r'\end{table}',
        '',
    ]
)

table_5 = [
    r'\begin{table}[htbp]',
    r'\centering',
    r'\caption{Development-frozen trading performance at a hypothetical cost of 0.01 per YES share.}',
    r'\label{tab:trading_evaluation}',
    r'\begin{tabular}{llrrrrr}',
    r'\toprule',
    r'Strategy & Block & Trades & Net PnL & Return & Hit rate & Max. drawdown \\',
    r'\midrule',
]
trading_table = trading.copy()
trading_table['block_order'] = trading_table['evaluation_block'].map({'INTERNAL_HOLDOUT': 0, 'EXTERNAL_TEST': 1})
for row in trading_table.sort_values(['strategy_label', 'block_order']).itertuples(index=False):
    table_5.append(
        f'{tex_escape(row.strategy_label)} & {tex_escape(row.evaluation_block_label)} & {int(row.trade_count)} & '
        f'{row.total_net_pnl:.4f} & {100.0 * row.return_on_capital:.1f}\% & '
        f'{100.0 * row.hit_rate:.1f}\% & {row.max_drawdown:.4f} \\\\'
    )
table_5.extend(
    [
        r'\bottomrule',
        r'\end{tabular}',
        r'\begin{minipage}{0.94\textwidth}',
        r'\footnotesize The observed pre-cutoff YES price is a fill proxy. Bid--ask spread, liquidity, slippage, partial fills and market impact are not modelled.',
        r'\end{minipage}',
        r'\end{table}',
        '',
    ]
)

tables_text = '\n'.join(table_1 + table_2 + table_3 + table_4 + table_5)
tables_path = REPORT_DIR / '18yB_results_tables.tex'
tables_path.write_text(tables_text, encoding='utf-8')

# ------------------------------------------------------------------
# Results statements
# ------------------------------------------------------------------
weather_min = weather['mean_error_c'].min()
weather_max = weather['mean_error_c'].max()
statements = [
    r'\paragraph{Sample and deterministic forecast diagnostics.}',
    (
        r'The final certified sample contains \NCertifiedContracts{} date-specific contracts arranged into '
        r'\NSettlementDates{} mutually exclusive and exhaustive contract books. '
        rf'Across the four decision rules, the deterministic forecast has mean error between {weather_min:.3f}$^\circ$C and {weather_max:.3f}$^\circ$C, '
        r'showing a persistent underforecast of the HKO daily maximum.'
    ),
    '',
    r'\paragraph{Development model selection.}',
    (
        r'The pooled empirical residual distribution achieves the lowest development CRPS '
        r'(\DevelopmentEmpiricalCRPS{}), compared with \DevelopmentRawCRPS{} for the raw deterministic forecast. '
        r'This corresponds to a \DevelopmentEmpiricalCRPSReduction\% reduction. '
        r'The Mat\'{e}rn-$3/2$ Gaussian process is the selected Gaussian-process comparator, while the pooled CatBoost quantile model is the selected tree comparator. '
        r'All selected probabilistic families under-cover the nominal 80\% interval on development data, with the CatBoost distribution displaying the greatest concentration.'
    ),
    '',
    r'\paragraph{Probability evaluation.}',
    (
        r'On the locked ten-date holdout, the uncalibrated empirical model records a categorical log score of '
        r'\HoldoutEmpiricalLogScore{}, compared with \HoldoutMarketLogScore{} for the normalised market. '
        r'This difference is reported descriptively and is not interpreted as statistically significant. '
        r'On the thirty-date June external block, the normalised market records \ExternalMarketLogScore{} and outperforms every weather-model variant; '
        r'the best weather-model value is \ExternalBestModelLogScore{} from the calibrated Mat\'{e}rn Gaussian process.'
    ),
    '',
    r'\paragraph{Trading evaluation.}',
    (
        r'The development-frozen primary strategy earns net PnL \PrimaryHoldoutPnL{} on the internal holdout but '
        r'\PrimaryExternalPnL{} in June, with maximum drawdowns \PrimaryHoldoutDrawdown{} and '
        r'\PrimaryExternalDrawdown{}, respectively. The Gaussian-process strategy also reverses sign out of time, '
        r'while the tree strategy is loss-making in both evaluation blocks. Hence positive holdout performance does not persist in June.'
    ),
    '',
    r'\paragraph{Evidential boundary.}',
    (
        r'The HKO availability convention uses an analytical working-day fallback rather than observed publication timestamps. '
        r'The forecast input is deterministic ECMWF IFS single-run data rather than a genuine ensemble. '
        r'Trading uses observed pre-cutoff YES prices as fill proxies and does not model spread, liquidity, slippage or market impact. '
        r'The small holdout and external samples therefore support a cautious empirical conclusion rather than a claim of stable arbitrage profitability.'
    ),
    '',
]
statements_text = '\n'.join(statements)
statements_path = REPORT_DIR / '18yB_results_statements.tex'
statements_path.write_text(statements_text, encoding='utf-8')

statement_inventory = pd.DataFrame(
    [
        ('sample_diagnostics', 'Sample and deterministic forecast diagnostics', 'Results: sample and forecast diagnostics'),
        ('development_selection', 'Development model selection', 'Results: post-processing comparison'),
        ('probability_evaluation', 'Probability evaluation', 'Results: market comparison'),
        ('trading_evaluation', 'Trading evaluation', 'Results: trading simulation'),
        ('evidential_boundary', 'Evidential boundary', 'Discussion or limitations'),
    ],
    columns=['statement_id', 'title', 'suggested_location'],
)

<>:87: SyntaxWarning: invalid escape sequence '\%'
<>:175: SyntaxWarning: invalid escape sequence '\%'
<>:176: SyntaxWarning: invalid escape sequence '\%'
<>:87: SyntaxWarning: invalid escape sequence '\%'
<>:175: SyntaxWarning: invalid escape sequence '\%'
<>:176: SyntaxWarning: invalid escape sequence '\%'
/var/folders/ck/dm6nl73d3v92cz_5d_bhjx_00000gn/T/ipykernel_57048/2224071386.py:87: SyntaxWarning: invalid escape sequence '\%'
  f'{row.date_balanced_mean_absolute_error_c:.3f} & {100.0 * row.date_balanced_interval_80_coverage:.1f}\% \\\\'
/var/folders/ck/dm6nl73d3v92cz_5d_bhjx_00000gn/T/ipykernel_57048/2224071386.py:175: SyntaxWarning: invalid escape sequence '\%'
  f'{row.total_net_pnl:.4f} & {100.0 * row.return_on_capital:.1f}\% & '
/var/folders/ck/dm6nl73d3v92cz_5d_bhjx_00000gn/T/ipykernel_57048/2224071386.py:176: SyntaxWarning: invalid escape sequence '\%'
  f'{100.0 * row.hit_rate:.1f}\% & {row.max_drawdown:.4f} \\\\'


In [4]:
# ------------------------------------------------------------------
# Synthesis figures
# ------------------------------------------------------------------
figure_rows = []

flow_map = {
    'Contract-decision candidates': 'Candidate rows',
    'Market-price-ready rows': 'Market ready',
    'Deterministic-weather-ready rows': 'Weather ready',
    'Exact market-weather common support': 'Exact common',
    'Complete-book contract rows': 'Complete-book rows',
}
flow = sample.loc[sample['object'].isin(flow_map)].copy()
flow['label'] = flow['object'].map(flow_map)
flow['order'] = flow['object'].map({name: order for order, name in enumerate(flow_map)})
flow = flow.sort_values('order', ascending=False)
fig, ax = plt.subplots(figsize=(8.5, 4.8))
bars = ax.barh(flow['label'], flow['count'])
ax.set_xlabel('Contract–decision rows')
ax.set_title('Final empirical support flow')
for bar, value in zip(bars, flow['count']):
    ax.text(value, bar.get_y() + bar.get_height() / 2, f' {int(value):,}', va='center')
fig.tight_layout()
path = FIGURE_DIR / '18yB_sample_support_flow.png'
fig.savefig(path, dpi=220, bbox_inches='tight')
plt.close(fig)
figure_rows.append(('fig:sample_support_flow', path, 'Final empirical support flow from candidate rows to complete-book contract rows.', 'Results: sample construction'))

model_plot = models.sort_values('date_balanced_mean_crps_c', ascending=False)
fig, ax = plt.subplots(figsize=(8.8, 5.8))
bars = ax.barh(model_plot['candidate_label'], model_plot['date_balanced_mean_crps_c'])
ax.set_xlabel('Date-balanced mean CRPS (°C)')
ax.set_title('Development-only residual-model comparison')
for bar, value in zip(bars, model_plot['date_balanced_mean_crps_c']):
    ax.text(value, bar.get_y() + bar.get_height() / 2, f' {value:.3f}', va='center')
fig.tight_layout()
path = FIGURE_DIR / '18yB_development_crps.png'
fig.savefig(path, dpi=220, bbox_inches='tight')
plt.close(fig)
figure_rows.append(('fig:development_crps', path, 'Development-only CRPS comparison for all nine frozen candidates.', 'Results: model selection'))

for block, file_stub, title in [
    ('INTERNAL_HOLDOUT', 'holdout_probability_log', 'Internal holdout: exact-common categorical log score'),
    ('EXTERNAL_TEST', 'external_probability_log', 'June external block: exact-common categorical log score'),
]:
    block_df = probability.loc[probability['evaluation_block'].eq(block)].sort_values('categorical_log_score', ascending=False)
    omitted_value = float(uncalibrated_catboost.loc[block, 'categorical_log_score'])
    fig, ax = plt.subplots(figsize=(8.8, 5.3))
    bars = ax.barh(block_df['method_label'], block_df['categorical_log_score'])
    ax.set_xlabel('Categorical log score (lower is better)')
    ax.set_title(title)
    for bar, value in zip(bars, block_df['categorical_log_score']):
        ax.text(value, bar.get_y() + bar.get_height() / 2, f' {value:.3f}', va='center')
    fig.subplots_adjust(bottom=0.16)
    fig.text(
        0.5,
        0.02,
        f'Uncalibrated CatBoost omitted from the compact scale: categorical log score {omitted_value:.3f}.',
        ha='center',
        fontsize=8,
    )
    path = FIGURE_DIR / f'18yB_{file_stub}.png'
    fig.savefig(path, dpi=220, bbox_inches='tight')
    plt.close(fig)
    label = 'fig:holdout_probability_log' if block == 'INTERNAL_HOLDOUT' else 'fig:external_probability_log'
    if block == 'INTERNAL_HOLDOUT':
        caption = (
            'Exact-common categorical log scores on the locked internal holdout. '
            f'The uncalibrated CatBoost value {omitted_value:.3f} is omitted from the compact scale and retained in the full audit table.'
        )
        location = 'Results: holdout probability comparison'
    else:
        caption = (
            'Exact-common categorical log scores on the June external block. '
            f'The uncalibrated CatBoost value {omitted_value:.3f} is omitted from the compact scale and retained in the full audit table.'
        )
        location = 'Results: external probability comparison'
    figure_rows.append((label, path, caption, location))

pnl_plot = trading.copy()
pnl_plot['plot_label'] = pnl_plot['strategy_label'] + ' — ' + pnl_plot['evaluation_block_label']
pnl_plot = pnl_plot.sort_values('total_net_pnl')
fig, ax = plt.subplots(figsize=(8.8, 5.0))
bars = ax.barh(pnl_plot['plot_label'], pnl_plot['total_net_pnl'])
ax.axvline(0.0, linewidth=1)
ax.set_xlabel('Net PnL at cost 0.01 per YES share')
ax.set_title('Development-frozen trading performance')
pnl_min = float(pnl_plot['total_net_pnl'].min())
pnl_max = float(pnl_plot['total_net_pnl'].max())
pnl_span = max(pnl_max - pnl_min, 1.0)
ax.set_xlim(pnl_min - 0.12 * pnl_span, pnl_max + 0.12 * pnl_span)
for bar, value in zip(bars, pnl_plot['total_net_pnl']):
    offset = 0.02 if value >= 0 else -0.02
    ha = 'left' if value >= 0 else 'right'
    ax.text(value + offset, bar.get_y() + bar.get_height() / 2, f'{value:.3f}', va='center', ha=ha)
fig.tight_layout()
path = FIGURE_DIR / '18yB_trading_net_pnl.png'
fig.savefig(path, dpi=220, bbox_inches='tight')
plt.close(fig)
figure_rows.append(('fig:trading_net_pnl', path, 'Net PnL of the three development-frozen strategies on the holdout and June blocks.', 'Results: trading performance'))

dd_plot = trading.copy()
dd_plot['plot_label'] = dd_plot['strategy_label'] + ' — ' + dd_plot['evaluation_block_label']
dd_plot = dd_plot.sort_values('max_drawdown')
fig, ax = plt.subplots(figsize=(8.8, 5.0))
bars = ax.barh(dd_plot['plot_label'], dd_plot['max_drawdown'])
ax.axvline(0.0, linewidth=1)
ax.set_xlabel('Maximum drawdown')
ax.set_title('Trading maximum drawdown from a zero initial portfolio value')
dd_min = float(dd_plot['max_drawdown'].min())
dd_max = float(dd_plot['max_drawdown'].max())
dd_span = max(dd_max - dd_min, 1.0)
ax.set_xlim(dd_min - 0.12 * dd_span, 0.05)
for bar, value in zip(bars, dd_plot['max_drawdown']):
    ax.text(value - 0.02, bar.get_y() + bar.get_height() / 2, f'{value:.3f}', va='center', ha='right')
fig.tight_layout()
path = FIGURE_DIR / '18yB_trading_maximum_drawdown.png'
fig.savefig(path, dpi=220, bbox_inches='tight')
plt.close(fig)
figure_rows.append(('fig:trading_maximum_drawdown', path, 'Maximum drawdown for the three development-frozen strategies, including the zero initial portfolio value.', 'Results or Discussion: trading risk'))

figure_inventory = pd.DataFrame(
    [
        {
            'figure_label': label,
            'path': str(path.relative_to(ROOT)),
            'caption': caption,
            'suggested_location': location,
            'sha256': sha256_file(path),
        }
        for label, path, caption, location in figure_rows
    ]
)

figure_snippets = []
for row in figure_inventory.itertuples(index=False):
    relative_from_report = Path(row.path).relative_to(REPORT_DIR.relative_to(ROOT))
    figure_snippets.extend(
        [
            r'\begin{figure}[htbp]',
            r'\centering',
            rf'\includegraphics[width=0.92\textwidth]{{{relative_from_report.as_posix()}}}',
            rf'\caption{{{tex_escape(row.caption)}}}',
            rf'\label{{{row.figure_label}}}',
            r'\end{figure}',
            '',
        ]
    )
figure_snippets_text = '\n'.join(figure_snippets)
figure_snippets_path = REPORT_DIR / '18yB_figure_snippets.tex'
figure_snippets_path.write_text(figure_snippets_text, encoding='utf-8')

table_inventory = pd.DataFrame(
    [
        ('tab:final_sample_flow', 'Final empirical sample flow and evaluation support', '18yB_results_tables.tex', 'Results: sample construction'),
        ('tab:development_model_selection', 'Development residual-model comparison', '18yB_results_tables.tex', 'Results: model selection'),
        ('tab:calibration_selection', 'Development-selected calibration parameters', '18yB_results_tables.tex', 'Methodology or Results: calibration'),
        ('tab:probability_evaluation', 'Exact-common probability evaluation', '18yB_results_tables.tex', 'Results: market comparison'),
        ('tab:trading_evaluation', 'Development-frozen trading performance', '18yB_results_tables.tex', 'Results: trading simulation'),
    ],
    columns=['table_label', 'caption', 'file', 'suggested_location'],
)

print('Thesis figures generated: PASS')
display(figure_inventory)

Thesis figures generated: PASS


,figure_label,path,caption,suggested_location,sha256
0,fig:sample_support_flow,reports/18yB_thesis_output_freeze/figures/18yB...,Final empirical support flow from candidate ro...,Results: sample construction,49a0487543d7699c29e85d67b1d5c03dbdcec53ca420bc...
1,fig:development_crps,reports/18yB_thesis_output_freeze/figures/18yB...,Development-only CRPS comparison for all nine ...,Results: model selection,1a8a451c1634c89392c3c74113091ce6aba6b62c676cb7...
2,fig:holdout_probability_log,reports/18yB_thesis_output_freeze/figures/18yB...,Exact-common categorical log scores on the loc...,Results: holdout probability comparison,92a383e7f459df421c80d0f35ca8894c3f28e4f8d38b9e...
3,fig:external_probability_log,reports/18yB_thesis_output_freeze/figures/18yB...,Exact-common categorical log scores on the Jun...,Results: external probability comparison,640971f6173ca2bb5c8f0d4c3b8905dd16d109fe3bd1d4...
4,fig:trading_net_pnl,reports/18yB_thesis_output_freeze/figures/18yB...,Net PnL of the three development-frozen strate...,Results: trading performance,726ac295af80a347d2863aba22a614cb11cbd54b50bdf1...
5,fig:trading_maximum_drawdown,reports/18yB_thesis_output_freeze/figures/18yB...,Maximum drawdown for the three development-fro...,Results or Discussion: trading risk,c49816e03ee00e5454c4f57d88cfe956c4d585dc573427...


In [5]:
# ------------------------------------------------------------------
# Freeze checks and outputs
# ------------------------------------------------------------------
check_rows: list[dict[str, Any]] = []
def add_check(check: str, passed: bool, detail: str) -> None:
    check_rows.append({'check': check, 'passed': bool(passed), 'detail': detail, 'blocking': True})

add_check('18yA_manifest_verified', manifest_entries > 0, f'entries={manifest_entries}')
add_check('latex_macros_created', macros_path.is_file(), str(macros_path.relative_to(ROOT)))
add_check('latex_tables_created', tables_path.is_file(), str(tables_path.relative_to(ROOT)))
add_check('latex_statements_created', statements_path.is_file(), str(statements_path.relative_to(ROOT)))
add_check('latex_figure_snippets_created', figure_snippets_path.is_file(), str(figure_snippets_path.relative_to(ROOT)))
add_check('thesis_tables_5', len(table_inventory) == 5, f'rows={len(table_inventory)}')
add_check('thesis_figures_6', len(figure_inventory) == 6, f'rows={len(figure_inventory)}')
add_check('statement_blocks_5', len(statement_inventory) == 5, f'rows={len(statement_inventory)}')
add_check('all_figures_exist', all((ROOT / path).is_file() for path in figure_inventory['path']), 'six PNG figures')
add_check('all_figures_nonempty', all((ROOT / path).stat().st_size > 10_000 for path in figure_inventory['path']), 'minimum 10 KB')
add_check('booktabs_present', all(token in tables_text for token in ['\\toprule', '\\midrule', '\\bottomrule']), 'booktabs table structure')
add_check('holdout_caution_present', 'not interpreted as statistically significant' in statements_text, 'ten-date holdout caveat')
add_check('market_fill_proxy_caution_present', 'fill proxy' in tables_text and 'slippage' in statements_text, 'execution boundary')
add_check('model_selection_not_rerun', True, '18yB depends only on frozen 18yA outputs')
add_check('probability_best_scores_bolded', '\\textbf{Pooled empirical residual (Uncalibrated)}' in tables_text and '\\textbf{Normalised market}' in tables_text, 'best method in each block is bolded')
add_check('catboost_omission_disclosed_in_table', 'uncalibrated CatBoost rows are omitted' in tables_text and f'{catboost_holdout_log:.3f}' in tables_text and f'{catboost_external_log:.3f}' in tables_text, 'compact table footnote records both omitted values')
add_check('catboost_omission_disclosed_in_figures', figure_inventory.loc[figure_inventory['figure_label'].isin(['fig:holdout_probability_log','fig:external_probability_log']), 'caption'].str.contains('uncalibrated CatBoost value', case=False).all(), 'both probability figure captions disclose omitted CatBoost')
add_check('sample_flow_caption_uses_contract_rows', 'complete-book contract rows' in figure_inventory.loc[figure_inventory['figure_label'].eq('fig:sample_support_flow'), 'caption'].iloc[0], 'caption matches plotted 3,850 contract rows')
add_check('sample_statement_uses_contract_books', 'date-specific contracts arranged into' in statements_text and 'mutually exclusive and exhaustive contract books' in statements_text, 'contract-book structure stated precisely')

integrity = pd.DataFrame(check_rows)
if not integrity['passed'].all():
    raise AssertionError('18yB blocking checks failed:\n' + integrity.loc[~integrity['passed']].to_string(index=False))

issues = pd.DataFrame(columns=['issue_level', 'issue_code', 'artefact', 'detail', 'blocking'])

table_inventory_path = OUT_DIR / '18yB_table_inventory.csv'
figure_inventory_path = OUT_DIR / '18yB_figure_inventory.csv'
statement_inventory_path = OUT_DIR / '18yB_statement_inventory.csv'
integrity_path = OUT_DIR / '18yB_integrity_checks.csv'
issues_path = OUT_DIR / '18yB_issues.csv'
table_inventory.to_csv(table_inventory_path, index=False)
figure_inventory.to_csv(figure_inventory_path, index=False)
statement_inventory.to_csv(statement_inventory_path, index=False)
integrity.to_csv(integrity_path, index=False)
issues.to_csv(issues_path, index=False)

source_inventory = pd.DataFrame(
    [
        {
            'input_role': path.stem,
            'path': str(path.relative_to(ROOT)),
            'rows': 1 if path.suffix == '.json' else len(pd.read_csv(path)),
            'sha256': sha256_file(path),
        }
        for path in REQUIRED
    ]
)
source_inventory_path = OUT_DIR / '18yB_source_inventory.csv'
source_inventory.to_csv(source_inventory_path, index=False)

freeze = {
    'step': STEP,
    'generated_at_utc': datetime.now(UTC).isoformat(),
    'verdict': 'PASS',
    'upstream_release': '18yA',
    'upstream_summary_sha256': sha256_file(A_SUMMARY),
    'upstream_manifest_sha256': sha256_file(A_MANIFEST),
    'model_selection_rerun': False,
    'calibration_selection_rerun': False,
    'trading_selection_rerun': False,
    'table_labels': table_inventory['table_label'].tolist(),
    'figure_labels': figure_inventory['figure_label'].tolist(),
    'statement_blocks': statement_inventory['statement_id'].tolist(),
    'verified_claim_rows': int(len(claims)),
    'evidential_boundary_rows': int(len(boundaries)),
    'presentation_revision': 'v2',
    'presentation_corrections': [
        'best probability scores bolded by evaluation block',
        'uncalibrated CatBoost omission disclosed',
        'sample-flow caption corrected to contract rows',
        'contract-book wording made precise',
    ],
    'results_outputs_frozen': True,
}
freeze_path = OUT_DIR / '18yB_thesis_output_freeze.json'
freeze_path.write_text(json.dumps(freeze, indent=2, ensure_ascii=False), encoding='utf-8')

protocol = {
    'step': STEP,
    'generated_at_utc': datetime.now(UTC).isoformat(),
    'verdict': 'PASS',
    'purpose': 'Render final dissertation tables, figures, macros and statements from the frozen 18yA evidence registry.',
    'upstream_dependency': '18yA only',
    'thesis_tables': int(len(table_inventory)),
    'thesis_figures': int(len(figure_inventory)),
    'results_statement_blocks': int(len(statement_inventory)),
    'model_selection_rerun': False,
    'calibration_selection_rerun': False,
    'trading_selection_rerun': False,
    'holdout_wording': 'descriptive; not interpreted as statistically significant',
    'external_wording': 'principal June out-of-time evaluation',
    'presentation_revision': 'v2',
    'probability_bold_rule': 'lowest score within each evaluation block',
    'uncalibrated_catboost_compact_omission_disclosed': True,
    'sample_flow_caption_unit': 'complete-book contract rows',
    'results_outputs_frozen': True,
}
protocol_path = OUT_DIR / '18yB_protocol.json'
protocol_path.write_text(json.dumps(protocol, indent=2, ensure_ascii=False), encoding='utf-8')

summary_b = {
    'step': STEP,
    'generated_at_utc': datetime.now(UTC).isoformat(),
    'verdict': 'PASS',
    'thesis_tables': int(len(table_inventory)),
    'thesis_figures': int(len(figure_inventory)),
    'results_statement_blocks': int(len(statement_inventory)),
    'latex_macro_commands': int(len(macro_values)),
    'verified_claim_rows': int(len(claims)),
    'evidential_boundary_rows': int(len(boundaries)),
    'results_outputs_frozen': True,
    'model_selection_rerun': False,
    'presentation_revision': 'v2',
    'uncalibrated_catboost_compact_omission_disclosed': True,
    'issue_rows': 0,
    'integrity_checks_passed': int(integrity['passed'].sum()),
    'integrity_checks_total': int(len(integrity)),
}
summary_path = OUT_DIR / '18yB_summary.json'
summary_path.write_text(json.dumps(summary_b, indent=2, ensure_ascii=False), encoding='utf-8')

environment = {
    'generated_at_utc': datetime.now(UTC).isoformat(),
    'python': sys.version,
    'platform': platform.platform(),
    'pandas': pd.__version__,
    'numpy': np.__version__,
    'matplotlib': __import__('matplotlib').__version__,
    'revision': 'v2',
}
environment_path = OUT_DIR / '18yB_environment.json'
environment_path.write_text(json.dumps(environment, indent=2), encoding='utf-8')

readme_lines = [
    '# 18yB dissertation output freeze',
    '',
    '**PASS**',
    '',
    '## LaTeX artefacts',
    '',
    '- `18yB_results_macros.tex`: frozen numeric macros.',
    '- `18yB_results_tables.tex`: five booktabs tables.',
    '- `18yB_results_statements.tex`: five concise dissertation-ready result blocks.',
    '- `18yB_figure_snippets.tex`: six figure environments.',
    '',
    '## Evidential wording',
    '',
    'The internal holdout is explicitly described as a ten-date descriptive check. '
    'June 2026 is the principal external out-of-time evaluation. The trading price is an observed pre-cutoff fill proxy, not an order-book execution quote.',
    '',
    '## Presentation revision v2',
    '',
    'Bold denotes the lowest probability score within each block. The compact probability table and figures disclose the omitted uncalibrated CatBoost log scores. The sample-flow caption refers to complete-book contract rows.',
]
readme_path = REPORT_DIR / '18yB_thesis_output_freeze_report.md'
readme_path.write_text('\n'.join(readme_lines) + '\n', encoding='utf-8')

manifest_rows = []
for root in [OUT_DIR, REPORT_DIR]:
    for path in sorted(root.rglob('*')):
        if path.is_file() and path.name != '18yB_sha256_manifest.csv':
            manifest_rows.append(
                {'path': str(path.relative_to(ROOT)), 'size_bytes': path.stat().st_size, 'sha256': sha256_file(path)}
            )
manifest_path = OUT_DIR / '18yB_sha256_manifest.csv'
pd.DataFrame(manifest_rows).to_csv(manifest_path, index=False)

print(json.dumps(summary_b, indent=2))
print('18yB dissertation output freeze release: PASS')

{
  "step": "18yB",
  "generated_at_utc": "2026-07-22T18:51:30.828537+00:00",
  "verdict": "PASS",
  "thesis_tables": 5,
  "thesis_figures": 6,
  "results_statement_blocks": 5,
  "latex_macro_commands": 17,
  "verified_claim_rows": 10,
  "evidential_boundary_rows": 10,
  "results_outputs_frozen": true,
  "model_selection_rerun": false,
  "presentation_revision": "v2",
  "uncalibrated_catboost_compact_omission_disclosed": true,
  "issue_rows": 0,
  "integrity_checks_passed": 19,
  "integrity_checks_total": 19
}
18yB dissertation output freeze release: PASS
